# Gender Coding Occupation

Maps free-text job titles to standardized occupation categories via LLM,
then attaches gender-composition data at the occupation and industry level.

All pipeline logic lives in `occupation_pipeline.py`. This notebook is the
configuration and driver.

In [1]:
from occupation_pipeline import (
    load_bls_occupation_map,
    load_jen_occupation_list,
    run_pipeline,
)

## 1. Define occupation sources

Both sources are loaded up front and passed to the pipeline.
The BLS source is a year-keyed dict; Jen's is a flat list.

In [ ]:
# BLS year-specific occupation lists
bls_map = load_bls_occupation_map("../data", year_range=range(2015, 2025))

# Jen's supplemental list
jen_list = load_jen_occupation_list("../data/supplemental/function_gender percentage.xlsx", "for Simon_cleaned", 'Job function')

occupation_sources = {
    # "bls": bls_map,
    "jen": jen_list,
}

## 2. Dataset configurations

Each dict describes one dataset: where it lives, which columns matter,
how to merge, and where to save. The  placeholder in
 is filled in per occupation source.

In [3]:
dataset_configs = [
    {
        "name": "admissions",
        "path": "../data/derived/admissions.csv",
        "date_col": "Round",
        "year_fixes": None,
        "drop_missing_year": False,
        "columns": [
            "Student ID",
            "Job 1 Organization",
            "Job #1 Industry Code",
            "Job 1 Title",
            "Round",
            "year",
        ],
        "id_name": "Student ID",
        "employer_col": "Job 1 Organization",
        "merge_on_idx": False,  # one row per student
        "output_path": "../data/derived/admissions_gendered_{source}.xlsx",
    },
    {
        "name": "interviews",
        "path": "../data/derived/interviews.csv",
        "date_col": "Application Date",
        "year_fixes": None,
        "drop_missing_year": False,
        "columns": [
            "Student ID",
            "Employer",
            "Industry",
            "Job Function",
            "Application Date",
            "year",
        ],
        "id_name": "Student ID",
        "employer_col": "Employer",
        "merge_on_idx": True,  # students can have multiple rows
        "output_path": "../data/derived/interviews_gendered_{source}.xlsx",
    },
    {
        "name": "outcomes",
        "path": "../data/derived/outcomes.csv",
        "date_col": "Offer Received Date",
        "year_fixes": {1923: 2023, 1918: 2018},
        "drop_missing_year": True,
        "columns": [
            "Student ID",
            "Employer",
            "Detailed Function",
            "Detailed Industry",
            "Reported Date",
            "Offer Received Date",
            "year",
        ],
        "id_name": "Student ID",
        "employer_col": "Employer",
        "merge_on_idx": True,  # students can have multiple rows
        "output_path": "../data/derived/outcomes_gendered_{source}.xlsx",
    },
]

## 3. Run the pipeline

This loops over each dataset × each occupation source, handles retries
automatically, and saves the results. Checkpointing means you can
interrupt and re-run without losing progress.

In [ ]:
run_pipeline(
    dataset_configs=dataset_configs,
    occupation_sources=occupation_sources,
    max_retries=7,
)


Processing: admissions

--- Occupation source: jen ---
  Processing: 1928 records (0 already done)


Processing:   0%|          | 0/1928 [00:00<?, ?it/s]

  Retry 1/7: 7 records (1921 already done)


Retry 1/7:   0%|          | 0/7 [00:00<?, ?it/s]

  ✅ All 1928 records processed.
  💾 Saved: ../data/derived/admissions_gendered_jen.xlsx


---

## Exploratory / diagnostic cells

Everything below is for inspection — not part of the production pipeline.

In [8]:
# Quick check: load one of the outputs and inspect
import pandas as pd

admissions_jen = pd.read_excel("../data/derived/interviews_gendered_jen.xlsx")
admissions_jen.head()

,Student ID,Job Title,Application Date,Graduation Year,Employer,Industry,Job Function,Type of Job,Employer Decision,Student OCI Status,Alternate List Position,OCI or Job Listings,Interview Date,year,census_occupation
0,220044010110147,Summer Consultant,2016-09-05,2018,Boston Consulting Group ( BCG ),Consulting - Strategy/Management,Consulting,Internship,Not Selected,Accepted Interview,NaN,OCI,2017-01-17,2016,Management Consulting
1,220044010110147,Summer Associate,2016-09-05,2018,Analysis Group Inc.,Consulting - Other,Consulting,Internship,Not Selected,Accepted Interview,NaN,OCI,2017-01-24,2016,Management Consulting
2,220044010110147,Summer Associate - Strategy & Operations,2016-09-05,2018,Deloitte,Consulting - Strategy/Management,Consulting,Internship,Not Selected,Accepted Interview,NaN,OCI,2017-01-23,2016,Management Consulting
3,220044010110164,Financial Management Associate-Summer Internship,2016-09-29,2019,Citi,Financial Services - Investment Banking/Brokerage,Finance - Corporate Finance,Internship,Extended Interview,Accepted Interview,NaN,OCI,2017-01-17,2016,Finance: Corporate Finance
4,220044010110164,MBA Summer Associate Program,2016-09-29,2019,Wayfair Inc,Technology - Internet Services/Telecomm,General Management,Internship,Not Selected,Accepted Interview,NaN,OCI,2017-01-20,2016,General Management


In [7]:
import pandas as pd
tst1 = pd.read_excel('../data/supplemental/function_gender percentage.xlsx')

In [8]:
tst2 = pd.read_excel('../data/supplemental/function_gender percentage.xlsx', sheet_name="for Simon_cleaned")

In [10]:
tst2.columns

Index(['Job Function', 'Average of func_gen_perc'], dtype='object')

In [17]:
func_set = set(tst1.func.values.tolist())
job_func_set = set(tst2['Job Function'].values.tolist())

In [18]:
len(job_func_set)

141

In [19]:
len(func_set)

240

In [20]:
job_func_set - func_set

{'Actor/Actress',
 'Communications',
 'Consulting: Other/E-commerce',
 'Customer Relationship Management (CRM)',
 'Economic Consulting',
 'Engineer',
 'Entrepreneur',
 'Founder',
 'Information Technology & Engineering',
 'Marketing & Business Development',
 'Marketing & Customer Relationship Management',
 'Merchandising & Purchasing',
 'Product Design',
 'Product Development',
 'Purchasing & Merchandising',
 'Real Estate (Commercial)',
 'Real Estate (Residential)',
 'Sales & Busines Development',
 'Sales: Account Management',
 'Sales: Customer Service',
 'Sales: E-Commerce',
 'Self-Employed',
 'Strategy & Business Development',
 'Strategy & Finance: Corporate',
 'Strategy & Finance: Corporate/Business Development',
 'Strategy: Product Management',
 'Trainer',
 'Volunteer: Board Member',
 'Volunteer: Non Profit',
 'Waiter/Waitress'}

In [21]:
func_set - job_func_set

{'.',
 'Accounting; Project Management',
 'Actor',
 'Actress',
 'Business Development & Marketing Strategy',
 'Business Development/Finance: Corporate',
 'Business Development/Finance: Investment Banking',
 'Business Development/Marketing/Strategy',
 'Communications/Public Relations',
 'Communications/Training',
 'Consulting - Other/E-commerce',
 'Customer Relations Mgmt',
 'Deceased',
 'Entrep/Founder/Self-Employed',
 'Entrepreneurial/Self-employed',
 'Finance - Corporate',
 'Finance - Inv Banking',
 'Finance - Investment Banking',
 'Finance:  Investment Banking',
 'Finance:  VC/PE',
 'Finance: Corporate/Business Development',
 'Finance: Corporate/Finance: Investment Banking',
 'Finance: Corporate/Information Technology',
 'Finance: Corporate/Strategic Planning',
 'Finance: Corporate/Strategy & Business Development',
 'Finance: Investment Banking/Risk Management',
 'Finance: Real Estate/General Management',
 'Finance: Risk Management and Human Resources',
 'Finance: Strategic Planning